# AI Research Agent

A research agent that answers questions about video games by combining a local
**RAG** knowledge base with a **web-search fallback**, and that **remembers** what it
learns from the web so it never has to search for the same thing twice.

**How it works, end to end:**

1. **Setup** — load API keys and configuration from `.env`.
2. **Phase 1 — RAG Pipeline** — embed a local games dataset into a persistent ChromaDB
   vector store and sanity-check semantic search.
3. **Phase 2 — Agent Tools** — wrap retrieval, a retrieval-quality judge, and a Tavily
   web-search fallback as callable tools.
4. **Phase 3 — Agent State Machine** — orchestrate the tools into a single agent that
   retrieves, evaluates confidence, falls back to the web when needed, and persists new
   knowledge back into the vector store.
5. **Phase 4 — Reporting** — turn the accumulated evidence into a structured, citable
   `ResearchReport`.
6. **Phase 5 — Testing & Validation** — exercise the agent against normal questions,
   a persistence check, and edge cases (ambiguous/misspelled/unanswerable questions).

Run the cells top to bottom in order — later phases depend on objects (`store`, `agent`,
`AGENT_TOOLS`, etc.) created in earlier ones.


# Phase 0 — Setup

Load the OpenAI and Tavily API keys from a local `.env` file (never committed — see
`.gitignore`) and verify they're present before anything else runs.


In [39]:
# Load in the OpenAI key and Tavily key.
# In the project folder, a file named '.env' has already been created.
# Ensure your .env file contains:
# OPENAI_API_KEY="your key"
# TAVILY_API_KEY="your key"
# OPENAI_BASE_URL="https://openai.vocareum.com/v1"

from dotenv import load_dotenv
import os

load_dotenv(".env")

assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("TAVILY_API_KEY") is not None

OPENAI_BASE_URL = os.getenv(
    "OPENAI_BASE_URL",
    "https://openai.vocareum.com/v1"
)

# Phase 1 — RAG Pipeline

Build a persistent ChromaDB vector store over the games dataset, embed each game as a document, and test semantic search before wiring it into the agent.

In [40]:
from lib.vector_store import VectorStoreManager
from lib.game_docs import load_games, build_documents

# Persists to AI_researchagent/chroma_db so the index survives notebook restarts.
store = VectorStoreManager(base_url=OPENAI_BASE_URL)
print(f"Collection '{store.collection_name}' ready at {store.persist_directory}, existing count={store.count()}")

Collection 'games' ready at C:\projects\AI_researchagent\chroma_db, existing count=35


In [41]:
# Load the game entries, build one natural-language document + metadata per game, and embed/upsert them.
games = load_games()
ids, documents, metadatas = build_documents(games)

store.add_documents(ids=ids, documents=documents, metadatas=metadatas)
print(f"Upserted {len(ids)} games. Collection count is now {store.count()}.")

Upserted 10 games. Collection count is now 35.


In [42]:
# Sanity-check semantic search standalone before wiring it into the agent as a tool.
sample_queries = [
    "Who developed FIFA 21?",
    "What is a good open-world RPG set in a medieval fantasy world?",
    "Which game follows Kratos and Atreus?",
]

for query in sample_queries:
    result = store.query(query, k=3)
    print(f"Query: {query}")
    for doc, meta, distance in zip(result["documents"][0], result["metadatas"][0], result["distances"][0]):
        print(f"  [{distance:.4f}] {meta['title']} — {doc}")
    print()

Query: Who developed FIFA 21?
  [0.2753] FIFA 21 — FIFA 21 is a Sports game developed by EA Vancouver and published by Electronic Arts. It was released on 2020-10-09 for PS4, Xbox One, PC, Nintendo Switch. FIFA 21 is a football simulation video game featuring updated squads, Career Mode, Volta Football, and Ultimate Team modes.
  [0.6558] Rockstar Games Is Working On A New Game... (It's Not GTA 6) — Rockstar Games Is Working On A New Game... (It's Not GTA 6). Remedy's Northlight engine, the same tech behind Control and Alan Wake 2. It's coming to PC, PlayStation 5, and Xbox Series XXS. The project is fully funded by Rockstar, and now we know they're actively collaborating during development. And as of today's update, the game is in full production, which is the phase where most of the core assets, levels, and systems get built out. What we don't have yet is a release date based on Remedy's typical QA development cycle. Plus, where they are now, late 
  [0.6635] Minecraft — Minecraft is

# Phase 2 — Agent Tools

Wrap the Phase 1 retrieval pipeline and a Tavily web-search fallback as `@tool`-decorated functions so the LLM can call them via function-calling.

In [43]:
from lib.agent_tools import (
    AGENT_TOOLS,
    configure_vector_store,
    retrieve_game,
    evaluate_retrieval,
    game_web_search,
)

# Reuse the persistent store from Phase 1 instead of letting the tools build their own.
configure_vector_store(store)

for t in AGENT_TOOLS:
    print(t.to_dict())


{'name': 'retrieve_game', 'description': 'Semantically search the games vector store for the top-k games most relevant to a natural-language question.\n\nReturns a JSON string: {"question": str, "results": [{"id", "title", "document", "metadata", "distance"}, ...]}.\nLower distance means a closer match.', 'parameters': {'type': 'object', 'properties': {'question': {'type': 'string'}, 'k': {'type': 'integer'}}, 'required': ['question']}}
{'name': 'evaluate_retrieval', 'description': 'Judge whether the results from `retrieve_game` are sufficient to answer the question.\n\nApplies a threshold on the best (lowest) distance score rather than a second LLM call, so the\nverdict is fast, deterministic, and cheap. `retrieved_results` must be the JSON string returned\nby `retrieve_game`. Returns a JSON string: {"sufficient": bool, "confidence": float, "reasoning": str}.', 'parameters': {'type': 'object', 'properties': {'question': {'type': 'string'}, 'retrieved_results': {'type': 'string'}, 'dis

In [44]:
# Tool 1 + Tool 2: retrieve locally, then judge whether the evidence is good enough to answer.
question = "Which game follows Kratos and Atreus?"

retrieval_json = retrieve_game.run(question=question, k=3)
print(retrieval_json)

verdict_json = evaluate_retrieval.run(question=question, retrieved_results=retrieval_json)
print(verdict_json)


{"question": "Which game follows Kratos and Atreus?", "results": [{"id": "god-of-war-ragnarok", "title": "God of War Ragnarok", "document": "God of War Ragnarok is a Action-Adventure game developed by Santa Monica Studio and published by Sony Interactive Entertainment. It was released on 2022-11-09 for PS4, PS5. God of War Ragnarok follows Kratos and Atreus as they journey through the Nine Realms to prepare for the prophesied Fimbulwinter.", "metadata": {"title": "God of War Ragnarok", "developer": "Santa Monica Studio", "platform": "PS4, PS5", "genre": "Action-Adventure", "publisher": "Sony Interactive Entertainment", "release_date": "2022-11-09"}, "distance": 0.3818233013153076}, {"id": "the-last-of-us-part-ii", "title": "The Last of Us Part II", "document": "The Last of Us Part II is a Action-Adventure game developed by Naughty Dog and published by Sony Interactive Entertainment. It was released on 2020-06-19 for PS4. The Last of Us Part II follows Ellie on a brutal journey of reven

In [45]:
# Tool 3: fall back to a live web search when a question is out of scope for the local games dataset.
fallback_question = "What is the latest patch version for Fortnite?"

web_results_json = game_web_search.run(query=fallback_question, max_results=3)
print(web_results_json)


{"query": "What is the latest patch version for Fortnite?", "results": [{"title": "37.20 Fortnite Ecosystem Updates and Release Notes | Fortnite Documentation | Epic Developer Community", "url": "https://dev.epicgames.com/documentation/fortnite/37-20-fortnite-ecosystem-updates-and-release-notes?lang=en-US", "snippet": "Fortnite v37.20 introduces the Roly Poly Spawner device, the Hive Stash Chest, and new prefabs and galleries from the O.X.R. Base and The Hive. This update also adds three new Debug Command menu options, new tutorials, and the ability to subscribe to input action events and mapping contexts directly through Verse.\n\n## New Experimental Character Inputs!\n\nYou can now subscribe to input action events and mapping contexts directly through Verse. The following character input actions are supported: ["}, {"title": "Fortnite News \u2013 Get the Latest Updates", "url": "https://www.fortnite.com/news", "snippet": "Aug 20, 2026. Fortnite Override: Break the Rules. Change the G

# Phase 3 — Agent State Machine / Orchestration

Wire `retrieve_game`, `evaluate_retrieval`, and `game_web_search` into a single `ResearchAgent`
that runs an explicit state machine:

`Start → Retrieve (RAG) → Evaluate → (if confident) Report`, or
`→ Web Search → Persist (upsert into ChromaDB) → Retrieve again → Report`.

A `max_iterations` guard prevents infinite loops if evidence never becomes sufficient.

In [46]:
from lib.agent import ResearchAgent

# Reuse the same persistent store/tools configured in Phase 1 & 2.
agent = ResearchAgent(store=store)


In [47]:
# Case 1: question is well covered by the local dataset -> should resolve via RAG only, no web search.
result = agent.answer("Who developed FIFA 21?")
print("\n".join(result["trace"]))
print("\nSources:", result["sources"])
print("\n" + result["report"].to_text())


[1] RETRIEVE: querying vector store for 'Who developed FIFA 21?'
[2] EVALUATE: sufficient=True confidence=0.7247

Sources: ['local knowledge base']

Q: Who developed FIFA 21?

Answer: FIFA 21 was developed by EA Vancouver.

Supporting details:
  - FIFA 21 is a sports game.
  - It was published by Electronic Arts.
  - The game was released on October 9, 2020.

Sources:
  - [internal knowledge] FIFA 21

Confidence: 1.00


In [48]:
# Case 2: question is NOT in the local dataset -> should fall back to web search, then persist the answer.
fallback_question = "What is Rockstar Games working on right now?"
result_web = agent.answer(fallback_question)
print("\n".join(result_web["trace"]))
print("\nSources:", result_web["sources"])
print("\n" + result_web["report"].to_text())


[1] RETRIEVE: querying vector store for 'What is Rockstar Games working on right now?'
[2] EVALUATE: sufficient=True confidence=0.6596

Sources: ['local knowledge base']

Q: What is Rockstar Games working on right now?

Answer: Rockstar Games is currently working on a new game that is not GTA 6, which is in full production with collaboration from Remedy Entertainment.

Supporting details:
  - The new game is being developed using Remedy's Northlight engine.
  - It is set to be released on PC, PlayStation 5, and Xbox Series X/S.
  - The project is fully funded by Rockstar and is in the phase where core assets, levels, and systems are being built.

Sources:
  - [web search] Rockstar Games Is Working On A New Game... (It's Not GTA 6) (https://www.youtube.com/watch?v=HeJZF23l4Ic)

Confidence: 0.80


In [49]:
# Re-ask the same question: it should now resolve from the vector store (long-term memory), no WEB_SEARCH step.
result_web_again = agent.answer(fallback_question)
print("\n".join(result_web_again["trace"]))
print("\nSources:", result_web_again["sources"])
print("\n" + result_web_again["report"].to_text())


[1] RETRIEVE: querying vector store for 'What is Rockstar Games working on right now?'
[2] EVALUATE: sufficient=True confidence=0.6596

Sources: ['local knowledge base']

Q: What is Rockstar Games working on right now?

Answer: Rockstar Games is currently working on a new game that is not GTA 6, which is in full production with collaboration from Remedy Entertainment.

Supporting details:
  - The new game is being developed using Remedy's Northlight engine.
  - It is fully funded by Rockstar and is in the phase where core assets, levels, and systems are being built.
  - The game is set to be released on PC, PlayStation 5, and Xbox Series X/S.

Sources:
  - [web search] Rockstar Games Is Working On A New Game... (It's Not GTA 6) (https://www.youtube.com/watch?v=HeJZF23l4Ic)

Confidence: 0.90


# Phase 4 — Reporting / Output Formatting

The REPORT step (`lib/report.py`) asks the LLM to synthesize the accumulated evidence — RAG hits,
web hits, and the evaluator's verdict — into a single structured `ResearchReport`:

- `answer` — a direct answer to the question
- `supporting_details` — short bullet-point facts backing the answer
- `sources` — every fact's origin, tagged `internal` (local knowledge base) or `web` (Tavily search)
- `confidence` — the model's overall confidence, 0.0-1.0

`ResearchAgent.answer(...)` returns this Pydantic model under the `"report"` key, so downstream
code can consume it programmatically (e.g. an API response) while `ResearchReport.to_text()`
renders it as readable text for the notebook.


In [50]:
# Inspect the structured report object directly: it's a plain Pydantic model, easy to serialize
# (e.g. `.model_dump_json()`) or hand off to an API response, in addition to `.to_text()` for display.
report = result_web["report"]

print(type(report))
print(report.model_dump_json(indent=2))


<class 'lib.report.ResearchReport'>
{
  "question": "What is Rockstar Games working on right now?",
  "answer": "Rockstar Games is currently working on a new game that is not GTA 6, which is in full production with collaboration from Remedy Entertainment.",
  "supporting_details": [
    "The new game is being developed using Remedy's Northlight engine.",
    "It is set to be released on PC, PlayStation 5, and Xbox Series X/S.",
    "The project is fully funded by Rockstar and is in the phase where core assets, levels, and systems are being built."
  ],
  "sources": [
    {
      "origin": "web",
      "title": "Rockstar Games Is Working On A New Game... (It's Not GTA 6)",
      "url": "https://www.youtube.com/watch?v=HeJZF23l4Ic"
    }
  ],
  "confidence": 0.8
}


# Phase 5 — Testing & Validation

Exercise the full agent end-to-end:

1. **Four example questions** — three should resolve purely from the local RAG store (FIFA 21 developer,
   God of War Ragnarok release date, Pokemon Red platform), and one has no static answer in the dataset
   (Rockstar Games' current project), forcing a `WEB_SEARCH` fallback.
2. **Persistence check** — re-ask the web-search-triggered question a second time and confirm it now
   resolves via RAG (no `WEB_SEARCH` step in the trace, and the vector store count no longer grows).
3. **Edge cases** — an ambiguous question, a misspelled game title, and a question with no answer even
   after a web search, all of which should produce a graceful "not found"-style answer instead of a
   hallucinated one.


In [51]:
# 1. Four example questions end-to-end: three should hit the local RAG store, one forces a web-search fallback.
test_questions = [
    "Who developed FIFA 21?",
    "When was God of War Ragnarok released?",
    "What platform was Pokemon Red released on?",
    "What is Rockstar Games' current project?",
]

for q in test_questions:
    count_before = store.count()
    result = agent.answer(q)
    count_after = store.count()
    print(f"{'=' * 80}\nQ: {q}")
    print("\n".join(result["trace"]))
    print(f"Sources: {result['sources']} | store count {count_before} -> {count_after}")
    print(result["report"].to_text())
    print()


Q: Who developed FIFA 21?
[1] RETRIEVE: querying vector store for 'Who developed FIFA 21?'
[2] EVALUATE: sufficient=True confidence=0.7247
Sources: ['local knowledge base'] | store count 35 -> 35
Q: Who developed FIFA 21?

Answer: FIFA 21 was developed by EA Vancouver.

Supporting details:
  - FIFA 21 is a sports game.
  - It was published by Electronic Arts.
  - The game was released on October 9, 2020.

Sources:
  - [internal knowledge] FIFA 21

Confidence: 1.00

Q: When was God of War Ragnarok released?
[1] RETRIEVE: querying vector store for 'When was God of War Ragnarok released?'
[2] EVALUATE: sufficient=True confidence=0.7561
Sources: ['local knowledge base'] | store count 35 -> 35
Q: When was God of War Ragnarok released?

Answer: God of War Ragnarok was released on November 9, 2022.

Supporting details:
  - Developed by Santa Monica Studio
  - Published by Sony Interactive Entertainment
  - Available on PS4 and PS5

Sources:
  - [internal knowledge] God of War Ragnarok

Confid

In [52]:
# 2. Persistence check: re-ask the web-search-triggered question and confirm it now resolves via RAG.
persistence_question = "What is Rockstar Games' current project?"

count_before_rerun = store.count()
result_rerun = agent.answer(persistence_question)
count_after_rerun = store.count()

print("Re-run trace:")
print("\n".join(result_rerun["trace"]))
print(f"Sources: {result_rerun['sources']} | store count {count_before_rerun} -> {count_after_rerun}")

assert "web search" not in result_rerun["sources"], "Expected the second run to resolve via RAG only"
assert count_after_rerun == count_before_rerun, "Store should not grow again on the RAG-only re-run"
print("\nPersistence verified: repeated question now resolves from the vector store without hitting the web.")


Re-run trace:
[1] RETRIEVE: querying vector store for 'What is Rockstar Games' current project?'
[2] EVALUATE: sufficient=True confidence=0.6646
Sources: ['local knowledge base'] | store count 35 -> 35

Persistence verified: repeated question now resolves from the vector store without hitting the web.


In [53]:
# 3. Edge cases: ambiguous question, misspelled title, and a question with no answer even after web search.
edge_case_questions = [
    "Tell me about the game.",                                # ambiguous / underspecified, no game named
    "Who developed Pokeman Rde?",                             # misspelled title
    "What is the release date of Xyzzyxville Quest 9000?",   # nonexistent game, no answer even on the web
]

for q in edge_case_questions:
    result = agent.answer(q)
    print(f"{'=' * 80}\nQ: {q}")
    print("\n".join(result["trace"]))
    print(f"Sources: {result['sources']}")
    print(result["report"].to_text())
    print()


Q: Tell me about the game.
[1] RETRIEVE: querying vector store for 'Tell me about the game.'
[2] EVALUATE: sufficient=False confidence=0.5027
[3] WEB_SEARCH: falling back to Tavily for 'Tell me about the game.'
[4] PERSIST: upserted 5 new document(s) into long-term memory
[5] RETRIEVE: querying vector store for 'Tell me about the game.'
[6] EVALUATE: sufficient=False confidence=0.5025
Sources: ['web search']
Q: Tell me about the game.

Answer: The evidence found does not answer the question about a specific video game.

Confidence: 0.20

Q: Who developed Pokeman Rde?
[1] RETRIEVE: querying vector store for 'Who developed Pokeman Rde?'
[2] EVALUATE: sufficient=False confidence=0.5869
[3] WEB_SEARCH: falling back to Tavily for 'Who developed Pokeman Rde?'
[4] PERSIST: upserted 5 new document(s) into long-term memory
[5] RETRIEVE: querying vector store for 'Who developed Pokeman Rde?'
[6] EVALUATE: sufficient=False confidence=0.587
Sources: ['web search']
Q: Who developed Pokeman Rde?

An

In [54]:
# Fix: tighten the REPORT prompt (lib/report.py) so it refuses to guess when the question doesn't
# name a specific game, or when the retrieved evidence doesn't actually describe a matching game.
# Reload the affected modules in-place so the running kernel picks up the change without a restart.
import importlib
import lib.report
import lib.agent

importlib.reload(lib.report)
importlib.reload(lib.agent)

from lib.agent import ResearchAgent

agent = ResearchAgent(store=store)

for q in edge_case_questions:
    result = agent.answer(q)
    print(f"{'=' * 80}\nQ: {q}")
    print("\n".join(result["trace"]))
    print(f"Sources: {result['sources']}")
    print(result["report"].to_text())
    print()


Q: Tell me about the game.
[1] RETRIEVE: querying vector store for 'Tell me about the game.'
[2] EVALUATE: sufficient=False confidence=0.5026
[3] WEB_SEARCH: falling back to Tavily for 'Tell me about the game.'
[4] PERSIST: upserted 5 new document(s) into long-term memory
[5] RETRIEVE: querying vector store for 'Tell me about the game.'
[6] EVALUATE: sufficient=False confidence=0.5027
Sources: ['web search']
Q: Tell me about the game.

Answer: The evidence found does not answer the question about a specific video game.

Confidence: 0.20

Q: Who developed Pokeman Rde?
[1] RETRIEVE: querying vector store for 'Who developed Pokeman Rde?'
[2] EVALUATE: sufficient=False confidence=0.5869
[3] WEB_SEARCH: falling back to Tavily for 'Who developed Pokeman Rde?'
[4] PERSIST: upserted 5 new document(s) into long-term memory
[5] RETRIEVE: querying vector store for 'Who developed Pokeman Rde?'
[6] EVALUATE: sufficient=False confidence=0.5868
Sources: ['web search']
Q: Who developed Pokeman Rde?

A

# Conclusion

The agent successfully answers questions from its local knowledge base via RAG, falls
back to a live web search when local evidence is insufficient, persists newly learned
facts into the vector store so repeat questions no longer hit the web, and produces a
structured, source-attributed `ResearchReport` for every answer. Edge cases (ambiguous
questions, misspellings, and genuinely unanswerable questions) are handled gracefully
instead of producing hallucinated answers.

**Possible next steps:** add re-ranking or a stronger evaluator for retrieval confidence,
support multi-turn conversations, and expand the games dataset.
